**Reference**: https://github-pages.ucl.ac.uk/rsd-engineeringcourse/ch07dry/

In [ ]:
import numpy as np

### functional paradigm

- map
- reduce
- lambda

**Nested functions** <br>
A conceptual trick which is often used by computer scientists to teach the core idea of functional programming is this: to write a program, in theory, you only ever need functions with one argument, even when you think you need two or more.

In [2]:
def get_numerator(numerator):
  def _get_denominator(denominator):
    return numerator/denominator
  return _get_denominator

division = get_numerator

In [6]:
print(get_numerator(20)(10))
print(division(35)(7))

2.0
5.0


**map**

- apply a function to each variable in an array, to return a new array.

In [ ]:
def exp_func(x):
    return 2**x

mylist = np.arange(0,10)
mylist = map(exp_func, mylist )
list (mylist)

[1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

**Reduce**

Accumulate values under an operation

In [9]:
from functools import reduce

def average(a, b):
  return (a+b)/2

def find_average(mylist):
  return reduce(average, mylist)

find_average([5,10,15,20,25])

20.3125

### Iterators

Iterators are useful when we don't want to allocate all values in the memory but instead use each value through iteration loops.

e.g., range() is an iterator

In [10]:
print(sum(range(0,10)))

# this save us from allocating million int list like below
for i in range(0, int(1e6)):
    i += i
print(i)

# we can also create list if we want
my_list = list(range(0, 5))
print(my_list)

45
1999998
[0, 1, 2, 3, 4]


The "iter" function lets us create iterator from any iterable object.

In [11]:
# the iter function
my_var = iter(range(3))
print(my_var)

print(next(my_var)) # 0
print(next(my_var)) # 1
print(next(my_var)) # 2
print(next(my_var)) # end of iterable

0
1
2


StopIteration: 

In [12]:
# string iterator
my_str = iter("abcd efg")
print(my_str)
for i in my_str:
  print(i)

a
b
c
d
 
e
f
g


###### Creating own variable

**Explicit Iterator** <br>
- use this if you need an object that maintains state, works with next(), and doesn’t reset each time

In [13]:
class fibonacci_iterator1:
  def __init__(self, max, seed1=1, seed2=1) -> None:
    self.max = max
    self.previous = seed1
    self.current = seed2

  def __iter__(self):
    return self

  def __next__(self):
    self.previous, self.current = self.current, self.previous + self.current
    self.max -= 1
    if self.max <= 0:
      raise StopIteration()
    return self.current

# now this class will act as iterator
my_iter_class = fibonacci_iterator1(10)
print(my_iter_class)
print(next(my_iter_class))
print(next(my_iter_class))

print("\nloop...")
for i in my_iter_class:
  print(i)

list(my_iter_class)

2
3

loop...
5
8
13
21
34
55
89


[]

**Generator-Based Iterable** <br>

- use this if you want a cleaner, memory-efficient, and reusable iterable that can be used multiple times.

In [14]:
class fibonacci_iterator2:
  def __init__(self, max, seed1=1, seed2=1) -> None:
    self.max = max
    self.previous = seed1
    self.current = seed2

  def __iter__(self):
    previous, current = self.current, self.previous + self.current
    for _ in range(self.max):
      yield current
      previous, current = current, previous + current

my_iter_generator = fibonacci_iterator2(10)
print(my_iter_generator)

for i in my_iter_generator:
  print(i)

list(my_iter_generator)

2
3
5
8
13
21
34
55
89
144


[2, 3, 5, 8, 13, 21, 34, 55, 89, 144]

### Generators

- use yield instead of returns
- maintain state

In [ ]:
def test_generator():
  yield "abc"
  yield "def"
  yield 23

my_var = test_generator()
print(my_var)

<generator object test_generator at 0x7d8c18e84460>


In [ ]:
print(next(my_var))
print(next(my_var))
print(next(my_var))
print(next(my_var))

abc
def
23


StopIteration: 

In [ ]:
def generate_odd_numbers(max):
  start = 1
  while start <= max:
    yield start
    start += 2

my_var = generate_odd_numbers(10)
print(my_var)

<generator object generate_odd_numbers at 0x7d8c193b0a00>


In [ ]:
for i in my_var:
  print(i)

1
3
5
7
9


In [ ]:
sum(my_var)

25

In [ ]:
# another example
def yield_fibonacci(limit, seed1, seed2):
  previous, current = seed1, seed2
  while limit > 0:
    limit -= 1
    yield current
    previous, current = current, previous + current

my_var = yield_fibonacci(10, 1, 1)
print(my_var)

sum(yield_fibonacci(10, 1, 1))

<generator object yield_fibonacci at 0x7d8c18f273d0>


231

### Context Managers

In [ ]:
class context_mgmt_example:
  def __init__(self) -> None:
    print("init sequence")

  def __enter__(self):
    print("enter sequence")
    return self

  def __exit__(self, exc_type, exc_val, exc_tb):
    print("exit sequence")


with context_mgmt_example() as cm:
  print("we do something inside the context")

init sequence
enter sequence
we do something inside the context
exit sequence


### Decorators

A decorator in Python is a higher-order function that allows us to modify or enhance the behavior of another function without changing its code.

In [ ]:
def my_decorator(func):
  def wrapper():
    print("something happens before function")
    func()
    print("something after the function")
  return wrapper

@my_decorator
def say_hello():
  print('hello')

say_hello()

something happens before function
hello
something after the function


In [ ]:
# using decorator to modify a function's return value without changing the original function
def upper_case_decorator(func):
  def _wrapper(*args, **kwargs):
    result = func(*args, **kwargs)
    return result.upper()
  return _wrapper

@upper_case_decorator
def greet(name):
  return f"Hello, {name}"

greet("Zephyr")

'HELLO, ZEPHYR'

In [ ]:
# another example
# here we modify the return values of our return_json function
# our original function return_json only returns data values
# but with wrapper, it also shows the status
# it returns an empty list, status is failed
# otherwise status is success

def jsonify(func):
  def _wrapper(*args, **kwargs):
    result = func(*args, **kwargs)
    if len(result) > 1:
      return {"status":"success", "data":result}
    else:
      return {"status":"failed", "data":result}
  return _wrapper

@jsonify
def return_json(input_list):
  my_dict = dict()
  for i,j in enumerate(input_list):
    my_dict[i] = j
  return my_dict


print(return_json([]))
print(return_json([1,23,2]))

{'status': 'failed', 'data': {}}
{'status': 'success', 'data': {0: 1, 1: 23, 2: 2}}


In [ ]:
# encrypting return value of function
import base64

def encrypt(func):
  def _wrapper(*args, **kwargs):
    result = func(*args, **kwargs)
    encoded_bytes = base64.b64encode(result.encode())
    return encoded_bytes.decode()
  return _wrapper

def decrypt(func):
  def _wrapper(*args, **kwargs):
    result = func(*args, **kwargs)
    decoded_bytes = base64.b64decode(result.encode())
    return decoded_bytes.decode()
  return _wrapper

# this encrypts the input
@encrypt
def plain_text_message(my_text):
    return my_text

plain_text_message("this is my secret message")

'dGhpcyBpcyBteSBzZWNyZXQgbWVzc2FnZQ=='

In [ ]:
@decrypt
@encrypt
def plain_text_message(my_text):
  return my_text

plain_text_message("this is another secret message")

'this is another secret message'

Alternative: Choosing When to Encrypt or Decrypt <br>
If you don’t always want decryption, you can use manual decorators instead

In [ ]:
message = encrypt(plain_text_message)("Original message")
print(message)

original = decrypt(plain_text_message)(message)
print(original)

T3JpZ2luYWwgbWVzc2FnZQ==
Original message


In [ ]:
# another example: repeating function calls
import random
import time

def repeater(max_retries=5, delays=2):
  def decorator(func):
    def _wrapper(*args, **kwargs):
      for attempt in range(1, max_retries+1):
        try:
          result = func(*args, **kwargs)
          if result is not None:
            print(f"Success on {attempt} attempt!")
            return result
          else:
            print(f"Attempt {attempt} returned None, Retrying...")
        except Exception as e:
          print(f"Error on {attempt} attempt: {e}")

        if attempt < max_retries:
          print(f"Retrying in {delays} seconds...")
          time.sleep(delays)

      print("All attempts failed.")
      return None

    return _wrapper
  return decorator

@repeater(max_retries=10, delays=2)
def unstable_network_call():
    if random.random() < 0.3:  # simulate 70% chance to fail
        print("Network Error! Retrying...")
        return None
    return "Success! Data retrieved."

unstable_network_call()

Network Error! Retrying...
Attempt 1 returned None, Retrying...
Retrying in 2 seconds...
Success on 2 attempt!


'Success! Data retrieved.'

In [ ]:
# the repeater is a generalized decorator so we can use it on other functions as well

@repeater(max_retries=3, delays=1)
def file_reader(filepath, mode):
  try:
    with open(filepath, mode) as file:
      return file.read()
  except FileNotFoundError:
    print("File not found. Retrying...")
    return None

file_reader("filename.csv", "r")

File not found. Retrying...
Attempt 1 returned None, Retrying...
Retrying in 1 seconds...
File not found. Retrying...
Attempt 2 returned None, Retrying...
Retrying in 1 seconds...
File not found. Retrying...
Attempt 3 returned None, Retrying...
All attempts failed.


### Exceptions

- we can define our custom Exceptions
- it is a good practice to use specific exception type instead of general exception

In [ ]:
# create a custom exception by inheriting from parent class
class MyCustomErrorType(ArithmeticError):
  pass

raise(MyCustomErrorType("this is my custom error"))

MyCustomErrorType: this is my custom error

In [ ]:
from sre_constants import CATEGORY
# we can add custom data to our exception
class MyCustomErrorType(Exception):
  def __init__(self, category=None):
    self.category = category

  def __str__(self):
    return f"Error, category {self.category}"

# raise(MyCustomErrorType(404))
# raise(MyCustomErrorType("abc"))

MyCustomErrorType: Error, category abc

In [ ]:
# handling exceptions with try except
import yaml

try:
  config = yaml.safe_load(open("datasource.yaml"))
  user = config["userid"]
  password = config["password"]

except FileNotFoundError:
  print("No password file is found")
  user = "anonymous"
  password = None

No password file is found


We can also manage multiple exceptions

In [ ]:
# setup example
with open('datasource2.yaml', 'w') as outfile:
  outfile.write("userid: eidle\n")
  outfile.write("password: 12345\n")

with open('datasource3.yaml', 'w') as outfile:
  outfile.write("user: eidle\n")
  outfile.write('password: secret\n')

In [ ]:
# function for reading the credential files
def read_credentials(sourcefile):
  try:
    with open(sourcefile, 'r') as source:
      config = yaml.safe_load(source)
      user = config["userid"]
      password = config["password"]
  except FileNotFoundError:
    print("Password file is missing")
    # set user as annonymous
    user = "annonymous"
    password = None
  except KeyError:
    print("Requested key name is missing")
    user = "annonymous"
    password = None
  finally:
    # this gets executed no matter what
    print(f"Trying to login with username: {user}...")
  return user, password

In [ ]:
print(read_credentials('datasource1.yaml'))

Password file is missing
Trying to login with username:annonymous...


('annonymous', None)

In [ ]:
print(read_credentials('datasource2.yaml'))

Trying to login with username: eidle...
('eidle', 12345)


In [ ]:
print(read_credentials('datasource3.yaml'))

Requested key name is missing
('annonymous', None)


Exceptions can be caught anywhere above the calling point in the call stack.

In the below scenario, the actual exceptions are always raised in f4, however, they are passed along and caught at highest matching except clause.

In [ ]:
def f4(x):
  if x == 0:
    return
  if x == 1:
    raise ArithmeticError()
  if x == 2:
    raise SyntaxError()
  if x==3:
    raise TypeError()
  if x==4:
    raise ValueError()

def f3(x):
  try:
    print("F3 Start")
    f4(x)
    print("F3 End")
  except ArithmeticError:
    print("Exception at F3")

def f2(x):
  try:
    print("F2 Start")
    f3(x)
    print("F2 End")
  except SyntaxError:
    print("Exception at F2")

def f1(x):
  try:
    print("F1 Start")
    f2(x)
    print("F1 End")
  except TypeError:
    print("Exception at F1")

In [ ]:
f1(0) # no error

F1 Start
F2 Start
F3 Start
F3 End
F2 End
F1 End


In [ ]:
f1(1)

F1 Start
F2 Start
F3 Start
Exception at F3
F2 End
F1 End


In [ ]:
f1(2)

F1 Start
F2 Start
F3 Start
Exception at F2
F1 End


In [ ]:
f1(3)

F1 Start
F2 Start
F3 Start
Exception at F1


In [ ]:
f1(4) # this exception is was not caught and will result in error

F1 Start
F2 Start
F3 Start


ValueError: 

Exceptions can be used as part of normal program execution flow instead of just limiting them for catching errors.

In [ ]:
from os import close
# file opening flow
import yaml

# create sample file
with open('example.yaml', 'w') as outfile:
  outfile.write("modelname: brillant\n")

# try-it-and-handle-exception approach
def analysis(source):
  try:
    name = source['modelname'] # consider source is a dictionary-like object
    print("case 1: ", end="")
  except TypeError:
    # if it wasn't a dictionary-like object
    try:
      # maybe it is a yaml filepath
      source_data = open(source, 'r')
      content = yaml.safe_load(source_data)
      name = content['modelname']
      source_data.close()
      print("case 2: ", end="")
    except IOError:
      # maybe it is already a raw YAML content
      source = yaml.safe_load(source)
      name = source['modelname']
      print("case 3: s", end="")
  print(name)

analysis("modelname: amazing") # input is already a yaml content
analysis("/content/example.yaml") # input is a file path

dict_source = {"modelname": "fantastic"}
analysis(dict_source) # input is a dict-like object

case 3: amazing
case 2: brillant
case 1: fantastic


**Throw-Low / Catch-High Principle**

1.	Throw exceptions as close as possible to where the error occurs (low-level functions or modules).
2.	Catch exceptions at a higher level where they can be properly handled (main logic or entry points).

In [ ]:
# 🔴 Bad approach Example
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Error: Cannot divide by zero.")
        return None  # Swallowing the error without informing the caller

result = divide(5, 0)
print(result)  # Outputs "Error: Cannot divide by zero." but doesn't propagate the issue

Error: Cannot divide by zero.
None


🔴 Why is this bad?
	•	The error is “hidden” because it’s handled too early.
	•	The caller gets None but may not realize a division error occurred.
	•	The error message is printed, but the program might continue incorrectly.

In [ ]:
# ✅ Good Approach

def divide(a, b):
    if b == 0:
        raise ZeroDivisionError("Division by zero is not allowed")
    return a / b  # Exception is thrown but not caught here

def main():
    try:
        result = divide(5, 0)
        print(f"Result: {result}")
    except ZeroDivisionError as e:
        print(f"Handling error in main: {e}")  # Handling at a higher level

main()

Handling error in main: Division by zero is not allowed


✅ Why is this good?
	•	The low-level function (divide) only raises the exception.
	•	The higher-level function (main) decides how to handle it.
	•	The error isn’t silently swallowed—it’s clear what went wrong.


### Operator Overloading

In [ ]:
# examples of overloaded operators

4 + 2 # here + is used to add two numbers
'abc' + 'def' # but here it is overloaded to concat two strings

34 * 2 # multiply 34 by 2
"me" * 2 # repeat the string

'meme'

We can overload the operators in our own classes to achieve similar results

In [ ]:
class Room:
  def __init__(self, name, area):
    self.name = name
    self.area = area

In [ ]:
small = Room("small", 9)
print(small)

The above default print output is not very helpful. We will define our own string representation by overloading it.

In [ ]:
### overloading string representation
class Room:
  def __init__(self, name, area):
    self.name = name
    self.area = area

  def __str__(self):
    # this will be default print output
    return f"<Room: {self.name} {self.area}m\u00b2>" # \u00b is for superscript

small = Room("small", 9)
print(small)

<Room: small 9m²>


In [ ]:
### overloading the addition and equality operator
class Room:
  def __init__(self, name, area):
    self.name = name
    self.area = area

  def __add__(self, other):
    return Room(f"{self.name}_&_{other.name}", self.area+other.area)

  def __eq__(self, other):
    return (self.area == other.area) and (self.name == other.name)

  def __str__(self):
    return f"<Room: {self.name} {self.area}m\u00b2>"

small1 = Room("small", 9)
large = Room("large", 20)
small2 = Room("small", 9)

# testing addition operator
merged_room = small2 + large
print(merged_room)

# testing equality operator
print(small1==small2)
print(small1==large)

<Room: small_&_large 29m²>
True
False


<table>
  <tr>
    <th>Operator</th>
    <th>Function</th>
  </tr>
  <tr>
    <td>Addition</td>
    <td>__add__</td>
  </tr>
  <tr>
    <td>Equality</td>
    <td>__eq__</td>
  </tr>
  <tr>
    <td> < </td>
    <td> __lt__ </td>
  </tr>
  <tr>
    <td> <= </td>
    <td> __le__ </td>
  </tr>
    <tr>
    <td> > </td>
    <td>__gt__</td>
  </tr>
  <tr>
    <td> >= </td>
    <td>__ge__</td>
  </tr>


</table>

In [ ]:
### overloading membership operator
### overloading len operator
### overloading item indexing

class Room:
  def __init__(self, name, area):
    self.name = name
    self.area = area
    self.occupants = []
  def add_occupant(self, name):
    self.occupants.append(name)
  def __contains__(self, value):
    # for checking membership (overload in operator)
    return value in self.occupants
  def __len__(self):
    return len(self.occupants)
  def __getitem__(self, index):
    return self.occupants[index]
  def __str__(self):
    return f"<Room: {self.name} {self.area}m\u00b2>"

observatory = Room("terra", 3)
observatory.add_occupant("Jerry")
observatory.add_occupant("Yuri")
observatory.add_occupant("Moris")

print(len(observatory)) # num of occupants
print("Yuri" in observatory.occupants)
print("Jerry" in observatory)
print(observatory[1], observatory[2])
print(observatory)

3
True
True
Yuri Moris
<Room: terra 3m²>


In [ ]:
### overloading call operator ()
### we can define classes that behave as functions

class Greeter(object):
  def __init__(self, greeting):
    self.greeting = greeting

  def __call__(self, name):
    print(f"{self.greeting}, {name}")

greeter_instance = Greeter("Hello")
greeter_instance("John")

Hello, John


### Metaprogramming

a technique where a program has the ability to treat itself or others as data. This means that the program can inspect, modify, or generate code at runtime or compile time.

Runtime metaprogramming: This happens during the execution of the program. The code can be dynamically altered or generated while the program is running. For example, in languages like Python or Ruby, you can use metaprogramming techniques to define new methods, modify classes, or manipulate the behavior of objects during runtime.

Some examples of metaprogramming
**Reflection**: The ability to inspect and modify the structure of a program during its execution. Many languages like Java, C#, and Python have reflection capabilities. <br>
**Code generation**: Creating code on the fly based on certain conditions, like generating getters and setters in a class automatically. <br>
**Dynamic method invocation**: Calling methods or accessing properties that may not be known until runtime.

In [ ]:
# metaprogram a class, by accessing its attribute dictionary
class Boring:
  pass

x = Boring()
print(x)

x.name = "Geoff"
print(x)
print(x.name)
print(x.__dict__)

# We can use getattr to access this special dictionary:
getattr(x, 'name')

# if we want to add an attribute given it's name as a string, we can use setattr
setattr(x, "age", 95)
print(x.age)

# since func are just variables that can be called with ()
# we can set attribute to a function and make it a member function
setattr(Boring, 'describe', lambda self: f"{self.name} is {self.age}")
print(x.describe())
print(x.describe)

Geoff
{'name': 'Geoff'}
95
Geoff is 95
<bound method <lambda> of <__main__.Boring object at 0x7c68c16c3590>>


This method is set as an attribute of the class, not the instance, so it is available to other instances of Boring:

In [ ]:
y = Boring()
y.name = "Terry"
y.age = 97
print(y.describe())

Terry is 97


We can also define a standalone function and then bind it to the class. Then its first argument automatically becomes self.

In [ ]:
def broken_birth_year(b_instance):
  import datetime
  current = datetime.datetime.now().year
  return current - b_instance.age

Boring.birth_year = broken_birth_year
print(x.birth_year())

1930


In [ ]:
# it is now also available in Y although y was not redefined afterwards
y.birth_year()

1928